In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)

seq_len = 3
d_model = 4

X = torch.randn(seq_len, d_model)

W_Q = torch.randn(d_model, d_model)
W_K = torch.randn(d_model, d_model)
W_V = torch.randn(d_model, d_model)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

scores = Q @ K.T
weights = F.softmax(scores, dim= 1)

output = weights @ V

print("X shape:", X.shape)
print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)
print("\nOutput:")
print(output)



X shape: torch.Size([3, 4])
Q shape: torch.Size([3, 4])
K shape: torch.Size([3, 4])
V shape: torch.Size([3, 4])

Output:
tensor([[-1.2644,  1.1882, -1.9180,  0.6323],
        [-0.2210,  0.0488,  0.4162, -0.0860],
        [-2.3008,  2.0238, -3.9304,  1.4040]])


In [7]:
# Step 2 — save K and V
K_cache = K.clone()
V_cache = V.clone()

print("\nCached K:")
print(K_cache)

print("\nCached V:")
print(V_cache)

# Step 3 — add token 4
X_new = torch.randn(1, d_model)

Q4 = X_new @ W_Q
K4 = X_new @ W_K
V4 = X_new @ W_V

# Append the new K/V:
K_cache = torch.cat([K_cache, K4], dim=0)
V_cache = torch.cat([V_cache, V4], dim=0)

print("\nUpdated K cache shape:", K_cache.shape)
print("Updated V cache shape:", V_cache.shape)

# Step 4 — generate token 4 using the cache
scores_cached = Q4 @ K_cache.T
weights_cached = F.softmax(scores_cached, dim=-1)

output_cached = weights_cached @ V_cache

print("\nCached output:")
print(output_cached)

#Step 5 — verify against normal computation
X_full = torch.cat([X, X_new], dim=0)

Q_full = X_full @ W_Q
K_full = X_full @ W_K
V_full = X_full @ W_V

scores_full = Q_full @ K_full.T
weights_full = F.softmax(scores_full, dim=-1)

output_full = weights_full @ V_full

output_normal = output_full[-1:]

print("\nNormal output for token 4:")
print(output_normal)

print("\nCached output:")
print(output_cached)

print("\nOutputs equal:")
print(torch.allclose(output_normal, output_cached, atol=1e-6))


Cached K:
tensor([[ 0.1199,  0.3067,  0.0819, -0.0952],
        [-4.1672,  1.7677, -0.5693,  6.8415],
        [ 0.5626,  0.4597, -0.6601, -0.3138]])

Cached V:
tensor([[-0.2457,  0.0072,  0.4317, -0.0555],
        [-2.5336,  2.2080, -4.3789,  1.5780],
        [ 0.0198,  0.4539,  0.2646, -0.3832]])

Updated K cache shape: torch.Size([4, 4])
Updated V cache shape: torch.Size([4, 4])

Cached output:
tensor([[-0.5439, -0.1935, -1.0119,  0.5950]])

Normal output for token 4:
tensor([[-0.5439, -0.1935, -1.0119,  0.5950]])

Cached output:
tensor([[-0.5439, -0.1935, -1.0119,  0.5950]])

Outputs equal:
True


In [9]:
# Step 6 — add causal masking
import torch
import torch.nn.functional as F

scores = Q @ K.T

causal_mask = torch.triu(
    torch.ones(seq_len, seq_len, dtype=torch.bool),
    diagonal=1
)

scores = scores.masked_fill(
    causal_mask,
    float("-inf")
)

weights = F.softmax(scores, dim=-1)

output = weights @ V

print("Causal mask:")
print(causal_mask)

print("\nAttention weights:")
print(weights)

print("\nOutput:")
print(output)

print("Normal output:")
print(output_normal)

print("\nCached output:")
print(output_cached)

print("\nOutputs equal:")
print(
    torch.allclose(
        output_normal,
        output_cached,
        atol=1e-6
    )
)

Causal mask:
tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])

Attention weights:
tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 3.0338e-06, 0.0000e+00],
        [3.8647e-02, 9.0481e-01, 5.6542e-02]])

Output:
tensor([[-0.2457,  0.0072,  0.4317, -0.0555],
        [-0.2458,  0.0073,  0.4317, -0.0555],
        [-2.3008,  2.0238, -3.9304,  1.4040]])
Normal output:
tensor([[-0.5439, -0.1935, -1.0119,  0.5950]])

Cached output:
tensor([[-0.5439, -0.1935, -1.0119,  0.5950]])

Outputs equal:
True


In [10]:
# Step 8 — autoregressive KV-cache loop
import torch
import torch.nn.functional as F

torch.manual_seed(42)

d_model = 4

W_Q = torch.randn(d_model, d_model)
W_K = torch.randn(d_model, d_model)
W_V = torch.randn(d_model, d_model)

# Pretend these are 4 tokens arriving one at a time
tokens = torch.randn(4, d_model)

K_cache = None
V_cache = None

for step in range(tokens.size(0)):

    # Only the new token is processed
    x_new = tokens[step:step + 1]

    Q_new = x_new @ W_Q
    K_new = x_new @ W_K
    V_new = x_new @ W_V

    # Add new K and V to cache
    if K_cache is None:
        K_cache = K_new
        V_cache = V_new
    else:
        K_cache = torch.cat([K_cache, K_new], dim=0)
        V_cache = torch.cat([V_cache, V_new], dim=0)

    # New token attends to everything in cache
    scores = Q_new @ K_cache.T
    weights = F.softmax(scores, dim=-1)
    output = weights @ V_cache

    print(f"\nStep {step + 1}")
    print("New Q shape:", Q_new.shape)
    print("New K shape:", K_new.shape)
    print("New V shape:", V_new.shape)
    print("K cache shape:", K_cache.shape)
    print("V cache shape:", V_cache.shape)
    print("Attention weights:", weights)
    print("Output shape:", output.shape)


Step 1
New Q shape: torch.Size([1, 4])
New K shape: torch.Size([1, 4])
New V shape: torch.Size([1, 4])
K cache shape: torch.Size([1, 4])
V cache shape: torch.Size([1, 4])
Attention weights: tensor([[1.]])
Output shape: torch.Size([1, 4])

Step 2
New Q shape: torch.Size([1, 4])
New K shape: torch.Size([1, 4])
New V shape: torch.Size([1, 4])
K cache shape: torch.Size([2, 4])
V cache shape: torch.Size([2, 4])
Attention weights: tensor([[0.9763, 0.0237]])
Output shape: torch.Size([1, 4])

Step 3
New Q shape: torch.Size([1, 4])
New K shape: torch.Size([1, 4])
New V shape: torch.Size([1, 4])
K cache shape: torch.Size([3, 4])
V cache shape: torch.Size([3, 4])
Attention weights: tensor([[0.0037, 0.4107, 0.5856]])
Output shape: torch.Size([1, 4])

Step 4
New Q shape: torch.Size([1, 4])
New K shape: torch.Size([1, 4])
New V shape: torch.Size([1, 4])
K cache shape: torch.Size([4, 4])
V cache shape: torch.Size([4, 4])
Attention weights: tensor([[9.9247e-01, 2.4636e-03, 4.6754e-03, 3.9539e-04]])
O

In [11]:
print("\nFinal K cache:")
print(K_cache)

print("\nFinal V cache:")
print(V_cache)


Final K cache:
tensor([[-1.0238, -0.4407, -0.1572, -0.9515],
        [-2.5754,  0.2765,  0.8850,  0.0684],
        [-2.5696,  0.2087,  0.5181, -0.0601],
        [-3.3947,  0.5268,  0.4546,  2.6657]])

Final V cache:
tensor([[ 0.2640,  2.2694, -0.7149, -1.3630],
        [ 3.2613, -2.4534,  2.0981, -0.9124],
        [ 1.1926,  2.5783, -0.4539, -2.7293],
        [ 0.8058,  1.0598, -0.0803, -1.5598]])
